# 第 9 章 プーリングと差分の差分: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/09-pooling-and-did.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 200              # 各群・各時点の標本の大きさ
effect = 1.5         # 処置の真の効果
entities = 100       # パネルの個人の数
periods = 5          # パネルの期間の数
correlation = 1.0    # 個人の効果 a と x の相関の強さ
repetitions = 200    # パネルを作り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 差分の差分

In [ ]:
import numpy as np

from lib.chapter09.did import (
    did_estimate,
    did_regression,
    draw_did_sample,
    group_period_means,
    panel_estimators,
)

sample = draw_did_sample(n, effect, np.random.default_rng(seed))
means = group_period_means(sample)
print(means.round(3))
print(f"差分の差分 {did_estimate(sample):.3f}（真の効果 {effect}）")
print(f"回帰の交差項の係数 {did_regression(sample).coefficients.iloc[-1]:.3f}")

## 図: 処置群・対照群の平均と反事実

In [ ]:
from lib.chapter09.plot import plot_did

plot_did(means)

## パネル: プーリング OLS と 1 階差分（真の傾きは 1）

In [ ]:
estimates = panel_estimators(
    entities, periods, correlation, repetitions, np.random.default_rng(seed)
)
print(estimates.agg(["mean", "std"]).round(3))

## 試してみよう

- `effect` を 0.0 にすると、差分の差分は 0.062 で、0 の近くになります。
  処置の効果がなければ、差分の差分も、ほぼ 0 です
- `correlation` を 0.0 にすると、プーリング OLS の平均は 1.496 から 1.000 になります。
  個人の効果が x と無相関なら、プーリングでも偏りません。1 階差分は、どちらでも 1.000
  です